In [1]:
import ee
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [5]:
import math
from datetime import datetime
from pathlib import Path
import pandas as pd
import ee


DEM_SOURCES = {
    "copernicus": "COPERNICUS/DEM/GLO30_2024_1",  # versión 2024, reemplaza al GLO30 original (deprecated)
    "srtm": "USGS/SRTMGL1_003",                    # clásico, puede tener huecos en zonas escarpadas
}


def get_terrain_profile_area(lat, lon, area_meters=56, dem_source="copernicus"):
    """
    Calcula elevación, pendiente (slope) y orientación (aspect) promedio
    sobre un área alrededor del punto (buffer_meters, default ~1 hectárea).

    dem_source: "copernicus" (default, recomendado) o "srtm"
    """
    if dem_source not in DEM_SOURCES:
        raise ValueError(f"dem_source debe ser uno de {list(DEM_SOURCES.keys())}")

    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        # COPERNICUS/DEM/GLO30 es una ImageCollection de tiles -> hay que mosaiquear
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')
    slope = ee.Terrain.slope(elevation).rename('slope')
    aspect_deg = ee.Terrain.aspect(elevation)

    # Aspect es un dato circular (0-360°) -> promediamos via seno/coseno,
    # no con una media aritmética directa (eso daría resultados incorrectos
    # cuando el área cruza el norte, ej. valores cerca de 0° y 360°).
    aspect_rad = aspect_deg.multiply(math.pi / 180)
    aspect_sin = aspect_rad.sin().rename('aspect_sin')
    aspect_cos = aspect_rad.cos().rename('aspect_cos')

    combined = elevation.addBands(slope).addBands(aspect_sin).addBands(aspect_cos)

    # Combinamos mean + stdDev en un solo reducer (más eficiente que dos
    # llamadas separadas a reduceRegion). sharedInputs=True hace que ambos
    # reducers usen las mismas bandas de entrada en vez de duplicarlas.
    combined_reducer = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.stdDev(), sharedInputs=True
    )

    stats = combined.reduceRegion(
        reducer=combined_reducer,
        geometry=region,
        scale=30,  # resolución nativa de ambos DEMs
        bestEffort=True
    ).getInfo()

    mean_aspect_rad = math.atan2(stats['aspect_sin_mean'], stats['aspect_cos_mean'])
    mean_aspect_deg = math.degrees(mean_aspect_rad)
    if mean_aspect_deg < 0:
        mean_aspect_deg += 360

    results = {
        'lat': lat,
        'lon': lon,
        'dem_source': dem_source,
        'elevation_m': stats['elevation_mean'],
        'elevation_std_m': stats['elevation_stdDev'],
        'slope_deg': stats['slope_mean'],
        'slope_std_deg': stats['slope_stdDev'],
        'aspect_deg': mean_aspect_deg,
        # Nota: no calculamos un "std" simple de aspect porque es un dato
        # circular (igual que el promedio) -> requeriría un cálculo distinto
        # (varianza circular). Si lo necesitas, avisame y lo agregamos.
    }
    return results


def save_terrain_profile(terrain_data, out_prefix="terrain_profile_data", output_dir="../databases"):
    """
    Guarda el perfil de terreno con timestamp en el nombre,
    igual que soil_profile: {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame([terrain_data])

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
    # Finca Matanza 7.300921,-73.009794
    terrain_data = get_terrain_profile_area(7.3297, -73.1867, dem_source="copernicus")

    out_path, df = save_terrain_profile(terrain_data)
    print(df)

CSV guardado en ../databases/terrain_profile_data-v260804203621.csv (1x8)
      lat      lon  dem_source  elevation_m  elevation_std_m  slope_deg  \
0  7.3297 -73.1867  copernicus  1066.430713        10.020476   0.386888   

   slope_std_deg  aspect_deg  
0              0  263.835785  


In [6]:
import math
from datetime import datetime
from pathlib import Path
import pandas as pd


DEM_SOURCES = {
    "copernicus": "COPERNICUS/DEM/GLO30_2024_1",  # versión 2024, reemplaza al GLO30 original (deprecated)
    "srtm": "USGS/SRTMGL1_003",                    # clásico, puede tener huecos en zonas escarpadas
}


def get_terrain_profile_area(lat, lon, area_meters=56, dem_source="copernicus"):
    """
    Calcula elevación, pendiente (slope) y orientación (aspect) promedio
    sobre un área alrededor del punto (buffer_meters, default ~1 hectárea).

    dem_source: "copernicus" (default, recomendado) o "srtm"
    """
    if dem_source not in DEM_SOURCES:
        raise ValueError(f"dem_source debe ser uno de {list(DEM_SOURCES.keys())}")

    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        # COPERNICUS/DEM/GLO30 es una ImageCollection de tiles -> hay que mosaiquear
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')
    slope = ee.Terrain.slope(elevation).rename('slope')
    aspect_deg = ee.Terrain.aspect(elevation)

    # Aspect es un dato circular (0-360°) -> promediamos via seno/coseno,
    # no con una media aritmética directa (eso daría resultados incorrectos
    # cuando el área cruza el norte, ej. valores cerca de 0° y 360°).
    aspect_rad = aspect_deg.multiply(math.pi / 180)
    aspect_sin = aspect_rad.sin().rename('aspect_sin')
    aspect_cos = aspect_rad.cos().rename('aspect_cos')

    combined = elevation.addBands(slope).addBands(aspect_sin).addBands(aspect_cos)

    # Combinamos mean + stdDev en un solo reducer (más eficiente que dos
    # llamadas separadas a reduceRegion). sharedInputs=True hace que ambos
    # reducers usen las mismas bandas de entrada en vez de duplicarlas.
    # Combinamos mean + stdDev + count en un solo reducer. El count es clave
    # para diagnosticar: si da 1 pixel, un stdDev de 0 es matemáticamente
    # correcto pero no significa "terreno uniforme", sino "muestra insuficiente".
    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.count(), sharedInputs=True)
    )

    stats = combined.reduceRegion(
        reducer=combined_reducer,
        geometry=region,
        scale=30,  # resolución nativa de ambos DEMs
        bestEffort=True
    ).getInfo()

    print(f"[DEBUG] stats crudos: {stats}")

    mean_aspect_rad = math.atan2(stats['aspect_sin_mean'], stats['aspect_cos_mean'])
    mean_aspect_deg = math.degrees(mean_aspect_rad)
    if mean_aspect_deg < 0:
        mean_aspect_deg += 360

    results = {
        'lat': lat,
        'lon': lon,
        'dem_source': dem_source,
        'elevation_m': stats['elevation_mean'],
        'elevation_std_m': stats['elevation_stdDev'],
        'slope_deg': stats['slope_mean'],
        'slope_std_deg': stats['slope_stdDev'],
        'aspect_deg': mean_aspect_deg,
        # Nota: no calculamos un "std" simple de aspect porque es un dato
        # circular (igual que el promedio) -> requeriría un cálculo distinto
        # (varianza circular). Si lo necesitas, avisame y lo agregamos.
    }
    return results


def save_terrain_profile(terrain_data, out_prefix="terrain_profile_data", output_dir="../databases"):
    """
    Guarda el perfil de terreno con timestamp en el nombre,
    igual que soil_profile: {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame([terrain_data])

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
    # Finca Matanza 7.300921,-73.009794
    terrain_data = get_terrain_profile_area(7.3297, -73.1867, dem_source="copernicus")

    out_path, df = save_terrain_profile(terrain_data)
    print(df)

[DEBUG] stats crudos: {'aspect_cos_count': 20, 'aspect_cos_mean': -0.107378423243794, 'aspect_cos_stdDev': 0, 'aspect_sin_count': 20, 'aspect_sin_mean': -0.9942182226360955, 'aspect_sin_stdDev': 0, 'elevation_count': 20, 'elevation_mean': 1066.4307127542336, 'elevation_stdDev': 10.020476150444086, 'slope_count': 20, 'slope_mean': 0.38688787817955017, 'slope_stdDev': 0}
CSV guardado en ../databases/terrain_profile_data-v260804203901.csv (1x8)
      lat      lon  dem_source  elevation_m  elevation_std_m  slope_deg  \
0  7.3297 -73.1867  copernicus  1066.430713        10.020476   0.386888   

   slope_std_deg  aspect_deg  
0              0  263.835785  


In [7]:
import math
from datetime import datetime
from pathlib import Path
import pandas as pd


DEM_SOURCES = {
    "copernicus": "COPERNICUS/DEM/GLO30_2024_1",  # versión 2024, reemplaza al GLO30 original (deprecated)
    "srtm": "USGS/SRTMGL1_003",                    # clásico, puede tener huecos en zonas escarpadas
}


def get_terrain_profile_area(lat, lon, area_meters=56, dem_source="copernicus"):
    """
    Calcula elevación, pendiente (slope) y orientación (aspect) promedio
    sobre un área alrededor del punto (buffer_meters, default ~1 hectárea).

    dem_source: "copernicus" (default, recomendado) o "srtm"
    """
    if dem_source not in DEM_SOURCES:
        raise ValueError(f"dem_source debe ser uno de {list(DEM_SOURCES.keys())}")

    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        # COPERNICUS/DEM/GLO30 es una ImageCollection de tiles -> hay que mosaiquear
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')
    slope = ee.Terrain.slope(elevation).rename('slope')
    aspect_deg = ee.Terrain.aspect(elevation)

    # Aspect es un dato circular (0-360°) -> promediamos via seno/coseno,
    # no con una media aritmética directa (eso daría resultados incorrectos
    # cuando el área cruza el norte, ej. valores cerca de 0° y 360°).
    aspect_rad = aspect_deg.multiply(math.pi / 180)
    aspect_sin = aspect_rad.sin().rename('aspect_sin')
    aspect_cos = aspect_rad.cos().rename('aspect_cos')

    combined = elevation.addBands(slope).addBands(aspect_sin).addBands(aspect_cos)

    # Combinamos mean + stdDev en un solo reducer (más eficiente que dos
    # llamadas separadas a reduceRegion). sharedInputs=True hace que ambos
    # reducers usen las mismas bandas de entrada en vez de duplicarlas.
    # Combinamos mean + stdDev + count en un solo reducer. El count es clave
    # para diagnosticar: si da 1 pixel, un stdDev de 0 es matemáticamente
    # correcto pero no significa "terreno uniforme", sino "muestra insuficiente".
    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.count(), sharedInputs=True)
    )

    stats = combined.reduceRegion(
        reducer=combined_reducer,
        geometry=region,
        scale=30,  # resolución nativa de ambos DEMs
        bestEffort=True
    ).getInfo()

    print(f"[DEBUG] stats crudos: {stats}")

    mean_aspect_rad = math.atan2(stats['aspect_sin_mean'], stats['aspect_cos_mean'])
    mean_aspect_deg = math.degrees(mean_aspect_rad)
    if mean_aspect_deg < 0:
        mean_aspect_deg += 360

    results = {
        'lat': lat,
        'lon': lon,
        'dem_source': dem_source,
        'elevation_m': stats['elevation_mean'],
        'elevation_std_m': stats['elevation_stdDev'],
        'slope_deg': stats['slope_mean'],
        'slope_std_deg': stats['slope_stdDev'],
        'aspect_deg': mean_aspect_deg,
        # Nota: no calculamos un "std" simple de aspect porque es un dato
        # circular (igual que el promedio) -> requeriría un cálculo distinto
        # (varianza circular). Si lo necesitas, avisame y lo agregamos.
    }
    return results


def save_terrain_profile(terrain_data, out_prefix="terrain_profile_data", output_dir="../databases"):
    """
    Guarda el perfil de terreno con timestamp en el nombre,
    igual que soil_profile: {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame([terrain_data])

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


def debug_slope_pixels(lat, lon, area_meters=56, dem_source="copernicus"):
    """
    Diagnóstico: devuelve los valores de slope de CADA pixel individual
    dentro del área, para verificar si realmente son todos iguales
    (terreno uniformemente inclinado) o si hay un problema de cálculo.
    """
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')
    slope = ee.Terrain.slope(elevation).rename('slope')

    combined = elevation.addBands(slope)

    pixel_values = combined.reduceRegion(
        reducer=ee.Reducer.toList(),
        geometry=region,
        scale=30,
        bestEffort=True
    ).getInfo()

    print(f"[DEBUG] elevation por pixel: {pixel_values['elevation']}")
    print(f"[DEBUG] slope por pixel: {pixel_values['slope']}")
    return pixel_values


if __name__ == "__main__":
    # Finca Matanza 7.300921,-73.009794
    terrain_data = get_terrain_profile_area(7.3297, -73.1867, dem_source="copernicus")

    out_path, df = save_terrain_profile(terrain_data)
    print(df)

[DEBUG] stats crudos: {'aspect_cos_count': 20, 'aspect_cos_mean': -0.107378423243794, 'aspect_cos_stdDev': 0, 'aspect_sin_count': 20, 'aspect_sin_mean': -0.9942182226360955, 'aspect_sin_stdDev': 0, 'elevation_count': 20, 'elevation_mean': 1066.4307127542336, 'elevation_stdDev': 10.020476150444086, 'slope_count': 20, 'slope_mean': 0.38688787817955017, 'slope_stdDev': 0}
CSV guardado en ../databases/terrain_profile_data-v260804204139.csv (1x8)
      lat      lon  dem_source  elevation_m  elevation_std_m  slope_deg  \
0  7.3297 -73.1867  copernicus  1066.430713        10.020476   0.386888   

   slope_std_deg  aspect_deg  
0              0  263.835785  
